# Module 0 Assignment Solution
## Neural Networks for Regression and Classification
### End-to-End Deep Learning with PyTorch

---

This notebook provides a complete, annotated solution to the M0 assignment.  
For every code block you will find:
- **WHY** — the conceptual motivation for this step in the data science lifecycle
- **HOW** — the technical implementation details
- **WHAT** — how to interpret the output

**Pipeline overview:**
```
RAW DATA
   ↓  Load & inspect
EXPLORATORY DATA ANALYSIS
   ↓  Understand distributions, correlations, missing values
PREPROCESSING
   ↓  Impute, encode, scale, split
MODEL DEFINITION (nn.Module)
   ↓  MLP architecture — from scratch, no wrappers
TRAINING + EARLY STOPPING
   ↓  20-epoch budget, best weights saved
EVALUATION
   ↓  Regression: RMSE, R², true-vs-predicted
   ↓  Classification: accuracy, confusion matrix, ROC/AUC, sample predictions
```

## Setup

**WHY**: We import all libraries at the top, so dependencies are immediately visible, and the notebook is self-contained.  
We fix random seeds, so every run produces the same results — essential for reproducible research.  
`device` auto-selects GPU when available; the code works identically on CPU.

In [ ]:
# Uncomment on Google Colab if packages are missing
# !pip install torch torchvision scikit-learn pandas matplotlib seaborn -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend; remove for interactive Jupyter
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

# ----- Reproducibility -----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ----- Device -----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1: Regression — Ames Housing

**Business problem**: Build an automated valuation model (AVM) that predicts residential sale prices.  
Accuracy is measured in RMSE (dollars) — lower is better.  
We use the Ames, Iowa dataset: 1,460 property sales, 79 features.

**Data science lifecycle steps covered**:  
Load → EDA → Preprocess → Split → Build MLP → Train → Evaluate

## Q1 — Load and Explore the Dataset

**WHY**: Before touching a model, we must understand the data.  
EDA reveals distribution shape, extreme outliers, missing-value patterns, and which features most strongly predict the target.  
These observations directly inform preprocessing choices.

In [ ]:
# HOW: sklearn's fetch_openml downloads the dataset from OpenML — no credentials needed.
# data_id=42165 is the canonical Ames Housing dataset.
print('Loading Ames Housing dataset from OpenML...')
ames = fetch_openml(data_id=42165, as_frame=True, parser='auto')
df   = ames.frame.copy()

print(f'Shape: {df.shape}')   # (1460, 80) — 79 features + SalePrice
df.head()

In [ ]:
# WHAT: Basic statistics about the target and missing-value landscape.
TARGET = 'SalePrice'
y_raw  = df[TARGET].astype(float)

print('=== Target: SalePrice ===')
print(f'  Min:    ${y_raw.min():>10,.0f}')
print(f'  Median: ${y_raw.median():>10,.0f}')
print(f'  Mean:   ${y_raw.mean():>10,.0f}')
print(f'  Max:    ${y_raw.max():>10,.0f}')
print(f'  Std:    ${y_raw.std():>10,.0f}')

print('\n=== Missing Values (top 15 columns) ===')
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].head(15).to_string())
print(f'\nColumns with > 40% missing: {(null_pct > 40).sum()}')

In [ ]:
# WHY (log transform): SalePrice is right-skewed — a few mansions have very high prices.
# Log-transforming compresses the scale so the model treats a $10k error on a $100k house
# similarly to a $100k error on a $1M house.  It also makes the target closer to Gaussian,
# which aligns with the MSE loss assumption.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: raw target
axes[0].hist(y_raw, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('SalePrice (raw)')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(y_raw.mean(),   color='red',    ls='--', label='Mean')
axes[0].axvline(y_raw.median(), color='orange', ls='--', label='Median')
axes[0].legend()

# Panel 2: log-transformed target
log_price = np.log1p(y_raw)
axes[1].hist(log_price, bins=50, color='seagreen', edgecolor='white', alpha=0.85)
axes[1].set_title('log(1 + SalePrice)')
axes[1].set_xlabel('log(SalePrice + 1)')
axes[1].set_ylabel('Count')
axes[1].axvline(log_price.mean(), color='red', ls='--', label='Mean')
axes[1].legend()

# Panel 3: top correlations
num_df   = df.select_dtypes(include=[np.number])
corr_vec = num_df.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top10    = corr_vec.head(10)
axes[2].barh(top10.index[::-1], top10.values[::-1], color='coral', edgecolor='white')
axes[2].set_title('Top 10 Feature Correlations')
axes[2].set_xlabel('|Pearson r| with SalePrice')
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('ames_eda.png', dpi=110, bbox_inches='tight')
plt.show()

# WHAT: The raw distribution is right-skewed; the log-transformed distribution is much
# more symmetric.  OverallQual and GrLivArea have the strongest correlations with price.
print('Top 5 correlated features:')
print(top10.head().to_string())

## Q2 — Preprocess and Prepare the Data

**WHY**: Neural networks cannot handle NaN values, string categories, or vastly different feature scales.  
Each preprocessing step addresses a specific failure mode:
- **Drop high-missing columns**: columns with >40% missing carry little signal and imputing them would introduce too much noise.
- **Median imputation**: robust to outliers compared to mean imputation.
- **Label encoding**: converts categorical strings to integers; simple and sufficient for tree-inspired MLP layers.
- **StandardScaler on train-only**: prevents data leakage — the scaler must not see test statistics during fitting.

In [ ]:
import numpy as np

# ---- 1. Separate features and target ----
X_raw = df.drop(columns=[TARGET]).copy()
y_all = np.log1p(df[TARGET].values.astype(np.float32))   # log1p transform

# ---- 2. Drop columns with > 40% missing ----
miss_frac     = X_raw.isnull().mean()
drop_cols     = miss_frac[miss_frac > 0.40].index.tolist()
X_raw         = X_raw.drop(columns=drop_cols)
print(f'Dropped {len(drop_cols)} high-missing columns: {drop_cols}')

# ---- 3. Separate numeric / categorical ----
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Numeric cols: {len(num_cols)},  Categorical cols: {len(cat_cols)}')

# ---- 4. Impute ----
X_raw[num_cols] = X_raw[num_cols].fillna(X_raw[num_cols].median())
X_raw[cat_cols] = X_raw[cat_cols].fillna('Missing')

# ---- 5. Label-encode categoricals ----
# HOW: LabelEncoder assigns a unique integer to each category string.
# The integer ordering is arbitrary but the MLP can learn non-linear relationships
# so this typically works well for categorical features with many levels.
for col in cat_cols:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))

X_all = X_raw.values.astype(np.float32)
print(f'Final feature matrix: {X_all.shape}')

In [ ]:
# ---- 6. Train / Val / Test split ----
# WHY: We need three non-overlapping sets:
#   train  — used for gradient updates
#   val    — used for early stopping decisions (no gradient updates)
#   test   — never seen until final evaluation; gives an unbiased performance estimate

X_tr_full, X_test, y_tr_full, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED)
X_train,   X_val,  y_train,   y_val  = train_test_split(
    X_tr_full, y_tr_full, test_size=0.20, random_state=SEED)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

# ---- 7. Scale features ----
# CRITICAL: fit only on X_train.  The validation and test sets must be transformed
# using training statistics to simulate deployment conditions.
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit + transform
X_val   = scaler.transform(X_val)          # transform only
X_test  = scaler.transform(X_test)         # transform only

# ---- 8. PyTorch tensors and DataLoaders ----
def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32))

X_tr, y_tr = to_tensor(X_train, y_train)
X_vl, y_vl = to_tensor(X_val,   y_val)
X_te, y_te = to_tensor(X_test,  y_test)

BATCH_REG   = 64
train_ldr_r = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_REG, shuffle=True)

print('DataLoaders ready.')

## Q3 — Build and Train the Regression MLP

**WHY — architecture choices**:
- **3+ hidden layers** give the network enough capacity to model non-linear interactions between 70+ features.
- **BatchNorm** normalises layer inputs, accelerating training and reducing sensitivity to initialisation.
- **Dropout** randomly zeros out neurons during training — this acts as an ensemble of smaller networks, reducing overfitting.
- **Single linear output neuron** with no activation function is correct for regression: the network should be free to predict any real value.

**WHY — early stopping**:  
We have only ~935 training examples.  With 20 epochs the model may memorise training noise.  
Early stopping monitors validation loss and halts when it stops improving, then restores the best weights.

In [ ]:
class EarlyStopping:
    """
    HOW: Tracks the best validation loss seen so far.
    If 'patience' consecutive epochs pass without improvement
    (by at least min_delta), sets self.stop = True.
    Saves the best model state dict so we can restore it after stopping.
    """

    def __init__(self, patience=5, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.stop       = False
        self.best_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            # Deep-copy the state dict so later updates don't overwrite it
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        """Load the best checkpoint back into model."""
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

print('EarlyStopping class defined.')

In [ ]:
class AmesRegressor(nn.Module):
    """
    4-layer MLP regressor.

    HOW — layer choices:
    - Linear(in → 256): first projection; width 256 gives ample capacity for ~70 features.
    - BatchNorm1d: normalises activations to zero mean / unit variance per batch.
    - ReLU: introduces non-linearity; avoids vanishing gradients.
    - Dropout(0.3): randomly zeros 30% of activations during training.
    - Final Linear(64 → 1): single output neuron, no activation → unrestricted real value.
    """

    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)   # regression output — no activation
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # remove trailing dim: (B,1) -> (B,)


model_reg = AmesRegressor(in_features=X_train.shape[1]).to(device)
n_params  = sum(p.numel() for p in model_reg.parameters() if p.requires_grad)
print(model_reg)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# HOW — training loop:
# Each epoch iterates over mini-batches.  For each batch:
#   1. Zero gradients (otherwise they accumulate across batches)
#   2. Forward pass: compute predictions
#   3. Compute MSE loss
#   4. Backward pass: compute gradients via autograd
#   5. Optimizer step: update weights
# After each epoch we evaluate on the validation set (no gradient computation)
# and call the early stopper.

NUM_EPOCHS = 20
LR         = 1e-3

optimizer_r  = torch.optim.Adam(model_reg.parameters(), lr=LR, weight_decay=1e-4)
criterion_r  = nn.MSELoss()
es_reg       = EarlyStopping(patience=5)

train_losses_r, val_losses_r = [], []

for epoch in range(NUM_EPOCHS):
    # ---- Training phase ----
    model_reg.train()    # enables Dropout and BatchNorm training mode
    batch_losses = []
    for X_b, y_b in train_ldr_r:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_r.zero_grad()             # 1. clear gradients
        pred = model_reg(X_b)               # 2. forward pass
        loss = criterion_r(pred, y_b)       # 3. MSE loss
        loss.backward()                     # 4. backprop
        optimizer_r.step()                  # 5. update weights
        batch_losses.append(loss.item())

    # ---- Validation phase ----
    model_reg.eval()     # disables Dropout; BatchNorm uses running stats
    with torch.no_grad():
        val_pred = model_reg(X_vl.to(device))
        v_loss   = criterion_r(val_pred, y_vl.to(device)).item()

    t_loss = np.mean(batch_losses)
    train_losses_r.append(t_loss)
    val_losses_r.append(v_loss)

    es_reg(v_loss, model_reg)
    marker = '  <- best' if es_reg.counter == 0 else ''
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS}  train MSE: {t_loss:.4f}  val MSE: {v_loss:.4f}{marker}')

    if es_reg.stop:
        print(f'Early stopping triggered at epoch {epoch+1}!')
        es_reg.restore_best(model_reg)
        break

print(f'\nBest validation MSE: {es_reg.best_loss:.4f}')

## Q4 — Evaluate the Regression Model

**WHY — these specific metrics**:
- **MSE curves**: diagnose overfitting (train loss ↓ while val loss ↑) or underfitting (both losses high and flat).
- **RMSE in dollars**: directly interpretable to a business stakeholder — "on average, predictions are $X off from the true price."
- **R²**: proportion of variance explained; 0 = useless baseline, 1 = perfect. Context: linear regression on this dataset typically achieves R² ≈ 0.85–0.90.
- **True vs. Predicted scatter**: reveals systematic bias — if points are above the diagonal, the model under-predicts; if below, it over-predicts. Asymmetric error patterns suggest the log-transform may not fully correct for skew.

In [ ]:
# ---- Predictions on test set ----
model_reg.eval()
with torch.no_grad():
    y_pred_log = model_reg(X_te.to(device)).cpu().numpy()

# Inverse-transform: go from log-scale back to dollars
# WHY: RMSE in log-space is hard to interpret.  We report in original dollars.
y_true_orig = np.expm1(y_te.numpy())
y_pred_orig = np.expm1(y_pred_log)

rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
r2   = r2_score(y_true_orig, y_pred_orig)

print(f'Test RMSE: ${rmse:>10,.0f}')
print(f'Test R²:   {r2:.4f}')

# ---- Plots ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Loss curves
best_ep = int(np.argmin(val_losses_r))
ax = axes[0]
ax.plot(train_losses_r, lw=2, color='steelblue', label='Train MSE')
ax.plot(val_losses_r,   lw=2, color='coral',     label='Val MSE')
ax.axvline(best_ep, ls='--', color='gray', lw=1.2, label=f'Best epoch ({best_ep+1})')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (log-price scale)')
ax.set_title('Regression Training Curves')
ax.legend(); ax.grid(alpha=0.3)

# Panel 2: True vs. Predicted
ax2 = axes[1]
ax2.scatter(y_true_orig, y_pred_orig, alpha=0.35, s=12, color='steelblue', label='Test samples')
lim = (min(y_true_orig.min(), y_pred_orig.min()) * 0.95,
       max(y_true_orig.max(), y_pred_orig.max()) * 1.05)
ax2.plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
ax2.set_xlim(lim); ax2.set_ylim(lim)
ax2.set_xlabel('True Sale Price ($)')
ax2.set_ylabel('Predicted Sale Price ($)')
ax2.set_title(f'True vs. Predicted\nRMSE=${rmse:,.0f}  R²={r2:.3f}')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regression_evaluation.png', dpi=110, bbox_inches='tight')
plt.show()

# WHAT: Points clustered tightly around the diagonal indicate good predictions.
# Systematic under-prediction of high-value homes (> $400k) is common — the training
# data has fewer luxury properties, so the model has less experience there.

---
# Part 2: Classification — MNIST

**Business problem**: Automate digit recognition from handwritten bank forms.  
Accuracy and per-class AUC are the key metrics.  The confusion pattern matters: misclassifying a '0' as '8' (visually similar) is different from misclassifying '1' as '7'.  

MNIST: 70,000 grayscale 28×28 images, 10 digit classes (0–9).

## Q5 — Load and Explore MNIST

**WHY**: Image datasets require different EDA than tabular data.  
We check class balance (MNIST is nearly balanced, but many real datasets are not),  
visualise representative examples to spot label quality issues,  
and confirm that normalisation is appropriate.

**HOW (normalisation)**:  
`Normalize((0.1307,), (0.3081,))` subtracts the MNIST global mean and divides by the global std.  
This centres pixel values near zero, which stabilises gradient flow.

In [ ]:
import torchvision
from torchvision import datasets, transforms

# HOW: torchvision.datasets.MNIST downloads automatically.
# transforms.ToTensor() converts PIL images to (C, H, W) tensors in [0, 1].
# Normalize shifts to approximately (-1, 1) using pre-computed MNIST statistics.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds_m     = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f'Full training samples : {len(full_train_ds)}')
print(f'Test samples          : {len(test_ds_m)}')
print(f'Image tensor shape    : {full_train_ds[0][0].shape}')   # (1, 28, 28)
print(f'Number of classes     : 10 (digits 0-9)')

In [ ]:
# WHY: Visualising a few samples per class ensures we understand what the model sees.
# It can also surface label errors or unusual handwriting styles.

# Collect 3 examples per digit class
class_examples = {c: [] for c in range(10)}
for img, label in full_train_ds:
    if len(class_examples[label]) < 3:
        class_examples[label].append(img)
    if all(len(v) == 3 for v in class_examples.values()):
        break

fig, axes = plt.subplots(3, 10, figsize=(14, 5))
for row in range(3):
    for col in range(10):
        ax = axes[row, col]
        ax.imshow(class_examples[col][row].squeeze(), cmap='gray')
        if row == 0:
            ax.set_title(str(col), fontsize=11, fontweight='bold')
        ax.axis('off')

plt.suptitle('MNIST — 3 samples per digit class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('mnist_samples.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Class distribution ----
all_labels   = [label for _, label in full_train_ds]
label_counts = pd.Series(all_labels).value_counts().sort_index()

plt.figure(figsize=(8, 3))
plt.bar(label_counts.index, label_counts.values, color='steelblue', edgecolor='white')
plt.xlabel('Digit Class'); plt.ylabel('Training Count')
plt.title('MNIST Training Set: Class Distribution')
plt.xticks(range(10))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('mnist_distribution.png', dpi=110, bbox_inches='tight')
plt.show()

print('Class counts:')
print(label_counts.to_string())
# WHAT: MNIST is nearly perfectly balanced (~5,900-6,700 per class).
# This means standard accuracy is a fair metric — no need for class weighting.

# ---- Train / Val split (85% / 15%) ----
n_train_m = int(0.85 * len(full_train_ds))
n_val_m   = len(full_train_ds) - n_train_m
train_ds_m, val_ds_m = random_split(
    full_train_ds, [n_train_m, n_val_m],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_CLS    = 128
train_ldr_m  = DataLoader(train_ds_m, batch_size=BATCH_CLS, shuffle=True,  num_workers=0)
val_ldr_m    = DataLoader(val_ds_m,   batch_size=BATCH_CLS, shuffle=False, num_workers=0)
test_ldr_m   = DataLoader(test_ds_m,  batch_size=BATCH_CLS, shuffle=False, num_workers=0)

print(f'\nTrain: {n_train_m},  Val: {n_val_m},  Test: {len(test_ds_m)}')

## Q6 — Build and Train the Classification MLP

**WHY — Flatten first**: Each MNIST image is a (1, 28, 28) tensor.  
An MLP expects a 1-D feature vector, so we flatten to 784 = 28×28 input dimensions.  
(A CNN would preserve spatial structure — that's a topic for Module 3.)

**WHY — 10 output neurons, no softmax in the model**:  
`nn.CrossEntropyLoss` combines `log_softmax` + negative log-likelihood internally.  
Applying `softmax` inside the model and then passing to `CrossEntropyLoss` would be a bug  
(it would compute softmax twice, destabilising gradients).

**WHY — Adam optimizer**:  
Adam adapts the learning rate per-parameter using first and second moment estimates.  
It converges faster than plain SGD on this size of dataset.

In [ ]:
class MNISTClassifier(nn.Module):
    """
    3-hidden-layer MLP for 10-class digit classification.

    Architecture: Flatten → 512 → 256 → 128 → 10
    Each hidden layer: Linear → BatchNorm → ReLU → Dropout

    HOW — width choice: 512 neurons in the first layer gives enough width
    to learn diverse low-level stroke patterns from 784 input pixels.
    Width decreases toward the output as information is progressively compressed.
    """

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()   # (B, 1, 28, 28) -> (B, 784)
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 10)   # 10 raw logits — no softmax here
        )

    def forward(self, x):
        return self.net(self.flatten(x))


model_cls = MNISTClassifier().to(device)
n_params_cls = sum(p.numel() for p in model_cls.parameters() if p.requires_grad)
print(model_cls)
print(f'\nTrainable parameters: {n_params_cls:,}')

In [ ]:
# HOW — tracking accuracy:
# logits.argmax(dim=1) gives the predicted class index for each sample in the batch.
# Comparing to the true labels and computing the fraction that match gives accuracy.

NUM_EPOCHS_M = 20

optimizer_c = torch.optim.Adam(model_cls.parameters(), lr=1e-3)
criterion_c = nn.CrossEntropyLoss()
es_cls      = EarlyStopping(patience=5)

train_losses_m, val_losses_m = [], []
train_accs_m,   val_accs_m   = [], []

for epoch in range(NUM_EPOCHS_M):
    # ---- Training phase ----
    model_cls.train()
    b_loss, b_correct, b_total = [], 0, 0
    for X_b, y_b in train_ldr_m:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_c.zero_grad()
        logits = model_cls(X_b)                         # forward pass
        loss   = criterion_c(logits, y_b)               # cross-entropy
        loss.backward()                                  # backprop
        optimizer_c.step()                               # weight update
        b_loss.append(loss.item())
        b_correct += (logits.argmax(1) == y_b).sum().item()
        b_total   += len(y_b)

    # ---- Validation phase ----
    model_cls.eval()
    v_loss, v_correct, v_total = [], 0, 0
    with torch.no_grad():
        for X_b, y_b in val_ldr_m:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits   = model_cls(X_b)
            v_loss.append(criterion_c(logits, y_b).item())
            v_correct += (logits.argmax(1) == y_b).sum().item()
            v_total   += len(y_b)

    tl = np.mean(b_loss);    vl = np.mean(v_loss)
    ta = b_correct / b_total; va = v_correct / v_total

    train_losses_m.append(tl); val_losses_m.append(vl)
    train_accs_m.append(ta);   val_accs_m.append(va)

    es_cls(vl, model_cls)
    marker = '  <- best' if es_cls.counter == 0 else ''
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS_M}  loss: {tl:.4f}/{vl:.4f}  acc: {ta:.4f}/{va:.4f}{marker}')

    if es_cls.stop:
        print(f'Early stopping at epoch {epoch+1}!')
        es_cls.restore_best(model_cls)
        break

print(f'\nBest validation loss: {es_cls.best_loss:.4f}')

## Q7 — Evaluate the Classification Model

**WHY — multiple evaluation views**:
- **Loss + accuracy curves**: diagnose training dynamics (are we overfitting?).
- **Confusion matrix**: reveals *which* classes are confused — business-critical for understanding error patterns.
- **ROC/AUC**: measures discriminability per class independent of threshold. AUC = 1.0 means perfect separation; AUC = 0.5 means no better than chance.  We use *one-vs-rest* to get a curve per digit.
- **Sample predictions**: ground-truth visual inspection of where the model succeeds and fails.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

best_ep_m = int(np.argmin(val_losses_m))

ax = axes[0]
ax.plot(train_losses_m, lw=2, color='steelblue', label='Train loss')
ax.plot(val_losses_m,   lw=2, color='coral',     label='Val loss')
ax.axvline(best_ep_m, ls='--', color='gray', lw=1.2, label=f'Best epoch ({best_ep_m+1})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Training and Validation Loss')
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(train_accs_m, lw=2, color='steelblue', label='Train accuracy')
ax2.plot(val_accs_m,   lw=2, color='coral',     label='Val accuracy')
ax2.axvline(best_ep_m, ls='--', color='gray', lw=1.2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.set_ylim(0.9, 1.01)
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_training_curves.png', dpi=110, bbox_inches='tight')
plt.show()

# WHAT: If val loss rises while train loss continues falling → overfitting.
# Early stopping should have captured the val loss minimum.

In [ ]:
# HOW: Collect all predictions and true labels from the test set in a single pass.
all_preds, all_true = [], []
model_cls.eval()
with torch.no_grad():
    for X_b, y_b in test_ldr_m:
        logits = model_cls(X_b.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(y_b.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

overall_acc = (all_preds == all_true).mean()
print(f'Test Accuracy: {overall_acc:.4f}  ({overall_acc*100:.2f}%)')

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'Confusion Matrix — Test Set  (accuracy = {overall_acc:.4f})',
             fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=110, bbox_inches='tight')
plt.show()

# WHAT: The diagonal holds correct predictions; off-diagonal cells are errors.
# Find the most confused pair:
cm_nodiag = cm.copy(); np.fill_diagonal(cm_nodiag, 0)
i, j = np.unravel_index(cm_nodiag.argmax(), cm_nodiag.shape)
print(f'Most confused pair: true={i} predicted as {j}  ({cm[i, j]} times)')

In [ ]:
# HOW — ROC one-vs-rest:
# For each digit class k, treat all other classes as the negative class.
# Compute the ROC curve using the softmax probability for class k as the score.
# AUC (area under the ROC curve) summarises discriminability in a single number.

# Collect softmax probabilities for all test samples
all_probs = []
model_cls.eval()
with torch.no_grad():
    for X_b, _ in test_ldr_m:
        probs = F.softmax(model_cls(X_b.to(device)), dim=1).cpu().numpy()
        all_probs.append(probs)

all_probs = np.vstack(all_probs)   # shape: (10000, 10)

# Binarise labels for one-vs-rest
y_bin = label_binarize(all_true, classes=list(range(10)))  # (10000, 10)

fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
auc_scores = []

for k in range(10):
    fpr, tpr, _ = roc_curve(y_bin[:, k], all_probs[:, k])
    auc_k       = auc(fpr, tpr)
    auc_scores.append(auc_k)
    ax.plot(fpr, tpr, color=colors[k], lw=1.6, label=f'Digit {k}  (AUC={auc_k:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — One-vs-Rest  (MNIST Test Set)')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=110, bbox_inches='tight')
plt.show()

macro_auc = np.mean(auc_scores)
print(f'Per-class AUC: {[f"{a:.4f}" for a in auc_scores]}')
print(f'Macro-average AUC: {macro_auc:.4f}')
# WHAT: AUC values close to 1.0 indicate near-perfect discrimination for that digit.
# The digit with the lowest AUC is the hardest to distinguish from all other classes.

In [ ]:
# HOW: We display a 4x8 grid from the test set.
# Green title = correct prediction, Red title = incorrect prediction.
# This gives an intuitive feel for the types of errors the model makes.

# Collect 32 test images with their predictions
sample_imgs, sample_true, sample_pred = [], [], []
model_cls.eval()
for X_b, y_b in DataLoader(test_ds_m, batch_size=64, shuffle=True):
    with torch.no_grad():
        preds = model_cls(X_b.to(device)).argmax(1).cpu()
    for i in range(len(X_b)):
        sample_imgs.append(X_b[i].squeeze().numpy())
        sample_true.append(y_b[i].item())
        sample_pred.append(preds[i].item())
        if len(sample_imgs) >= 32:
            break
    if len(sample_imgs) >= 32:
        break

fig, axes = plt.subplots(4, 8, figsize=(14, 8))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_imgs[idx], cmap='gray')
    correct = (sample_true[idx] == sample_pred[idx])
    color   = 'green' if correct else 'red'
    ax.set_title(f'T:{sample_true[idx]}  P:{sample_pred[idx]}',
                 color=color, fontsize=8, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Test Predictions  (green = correct, red = wrong)', fontsize=12)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=110, bbox_inches='tight')
plt.show()

# WHAT: Incorrect predictions are often visually ambiguous even to humans —
# a poorly-formed '4' can look like a '9', or a loopy '7' can resemble a '1'.

---
## Summary and Reflection Answers

### 1. Data Leakage (StandardScaler)
Fitting `StandardScaler` on the full dataset before splitting means the scaler has seen the test set's mean and variance.  During evaluation, test features are centred using information they "shouldn't" have contributed.  This inflates test performance, because the model effectively memorised statistical properties of the test distribution.  In deployment, new data arrives without those statistics, so reported performance is over-optimistic.

### 2. Log Transform
Sale prices span nearly an order of magnitude ($35k–$755k).  MSE penalises proportionally to the square of the error in absolute terms, so a $100k error on a $500k house and a $100k error on a $150k house are treated identically.  After `log1p`, a $100k error on a $500k house (log ratio ≈ 0.18) is appropriately smaller than the same error on a $150k house (log ratio ≈ 0.47).  The log-transformed distribution is also closer to Gaussian, which is the implicit assumption of MSE loss.

### 3. Early Stopping
See printed output above.  Triggering before 20 epochs means the model saturates the training signal well before the budget is exhausted — either because the dataset is small relative to model capacity, or because the chosen architecture learns quickly.  If early stopping never triggered, the model either needed more epochs or the patience was too high.

### 4. Confusion Patterns
MNIST's most commonly confused pairs are typically **(4, 9)** and **(3, 5)** and **(7, 1)**.  These share similar stroke geometry — both '4' and '9' have a closed upper loop in some handwriting styles.  A convolutional network (CNN) would reduce these errors by exploiting local spatial structure rather than treating all pixels independently as an MLP does.

### 5. Business Decision
Before deploying at 97% accuracy you would want: (a) the *cost matrix* — what are the consequences of each error type (e.g., misreading a check amount vs. misreading a form field)?  (b) performance on the *deployment distribution* — is the test set representative of the production data (scanner quality, paper type, ink density)?  (c) a *monitoring plan* — how will accuracy be tracked in production?  (d) a *fallback policy* — what happens when the model's confidence is low?  97% means ~300 errors per 10,000 checks; that may or may not be acceptable depending on the downstream cost.